# Colab â€” large-scale training (maximize test-set macro-F1)
### PS7 exoplanet vetting pipeline

Runs the heavy work on Colab (fast MAST network): build a **~2,400-target** clean-label set (confirmed planets / TESS-EB catalog / field-star 'other'), extract **features + dual-view tensors** in parallel, **tune + train** the LightGBM and the augmented CNN, and report **cross-validated macro-F1** plus a **feature-ablation**. Download the trained models back into the repo's `data/features/`.

> The model is small; the bottleneck is downloads â€” Colab's datacenter pipe to MAST is the main speedup. A GPU is optional.

## 1. Setup â€” install deps + get the package

In [ ]:
import os, sys

# On Colab the VM is remote — local project files are not present.
# Auto-clone the repo so the exopipeline package is available.
REPO = 'https://github.com/veritas1407/Exoplanets-detection-TESS-Pipeline.git'
REPO_DIR = 'Exoplanets-detection-TESS-Pipeline'

if not os.path.isdir('exopipeline') and not os.path.isdir(REPO_DIR):
    os.system(f'git clone -q {REPO}')

if os.path.isdir(REPO_DIR) and not os.path.isdir('exopipeline'):
    os.chdir(REPO_DIR)

assert os.path.isdir('exopipeline'), f'Clone failed or wrong directory. cwd={os.getcwd()}'
print('cwd:', os.getcwd())

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import subprocess
subprocess.run([
    'pip', '-q', 'install',
    'numpy<2', 'astropy<7', 'lightkurve', 'transitleastsquares', 'wotan',
    'batman-package', 'emcee', 'corner', 'lightgbm', 'scikit-learn',
    'imbalanced-learn', 'astroquery', 'pyarrow', 'torch',
], capture_output=True)

import warnings; warnings.filterwarnings('ignore')
from exopipeline import labels, featurize, classify, cnn, config
import numpy as np, pandas as pd
print('ready — exopipeline imported')

## 2. Build the large clean-label set (~2,400 targets)
transit = confirmed CP/KP Â· eclipsing_binary = TESS EB catalog Â· other = TOI false-alarms + random field stars. Parallel build of features **and** views, checkpointed/resumable.

In [ ]:
PER_CLASS = 800     # ~2,400 total; lower for a quick run
sample = labels.build_clean_sample(per_class=PER_CLASS)
print('sample:', sample['label'].value_counts().to_dict(), '| total', len(sample))
feat, Xg, Xl, y, tics = featurize.build_training_set(sample, n_workers=os.cpu_count())
print('features:', feat.shape, '| views:', Xg.shape,
      '| classes:', dict(zip(*np.unique(y, return_counts=True))))

## 3. Tabular LightGBM â€” CV macro-F1 + hyperparameter tuning (O4)

In [ ]:
cv0 = classify.cross_validate(feat)
print(f'default CV macro-F1: {cv0["mean"]:.3f} +/- {cv0["std"]:.3f}')
best, best_score, _ = classify.tune_lightgbm(feat)   # extended grid (num_leaves up to 127)
print('tuned CV macro-F1:', round(best_score,3), '| params:', best)

# k-fold bagging: save 5 fold models -> predict() averages them (Schanche+2020)
cv_bag = classify.cross_validate(feat, n_splits=5, params=best, save_fold_models=True)
print(f'5-fold bagged CV macro-F1: {cv_bag["mean"]:.3f} +/- {cv_bag["std"]:.3f}')

res = classify.train(feat, params=best)
classify.save_model(res['model'])   # single calibrated model kept as fallback
print('PR-AUC:', round(res['pr_auc'],3))
print(pd.DataFrame(res['confusion_matrix'], index=res['classes'], columns=res['classes']))

## 4. Feature ablation â€” do the new SOTA features help?
Compare CV macro-F1 with the original 15 features vs the full 21 (adds stellar-density consistency, V-shape, transit/secondary SNR, per-transit scatter, phase coverage).

In [ ]:
OLD = ['period','depth_ppm','duration_hr','sde','snr','log_fap','odd_even_diff_ppm',
       'odd_even_sigma','secondary_ppm','flatness','n_transits','snr_per_transit',
       'dur_over_period','rp_rs','crowdsap']
import copy
_full = config.FEATURE_COLUMNS
config.FEATURE_COLUMNS = OLD
cv_old = classify.cross_validate(feat)
config.FEATURE_COLUMNS = _full
cv_new = classify.cross_validate(feat)
print(f'15 original features: CV macro-F1 = {cv_old["mean"]:.3f}')
print(f'21 full features    : CV macro-F1 = {cv_new["mean"]:.3f}')
print('delta:', round(cv_new['mean']-cv_old['mean'],3))

## 5. Dual-view CNN â€” with augmentation (O4 Track B)

In [ ]:
# Deeper dual-view CNN (residual blocks) + phase-mirror augmentation.
# k-fold bagging: 5 fold models saved to cnn_fold_*.pt -> predict averages them (with TTA).
cnn.train_cnn_cv(Xg, Xl, y, n_splits=5, n_epochs=200, batch_size=32, augment=True)
# single fallback model (used if fold files are absent)
rescnn = cnn.train_cnn(Xg, Xl, y, n_epochs=200, batch_size=32, augment=True)
print(pd.DataFrame(rescnn['confusion_matrix'], index=rescnn['classes'], columns=rescnn['classes']))
print('per-class:'); print(pd.DataFrame(rescnn['report']).T[['precision','recall','f1-score']].round(3))

## 6. Leak-free ensemble + tune the fusion weight
Both tracks trained on the SAME split; evaluate Tabular / CNN / Ensemble on unseen data and pick the fusion weight that maximizes held-out macro-F1.

In [ ]:
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
import lightgbm as lgb
view = {t:i for i,t in enumerate(tics)}
fa = feat.dropna(subset=['label','target']).drop_duplicates('target')
fa = fa[fa['target'].isin(view)].reset_index(drop=True)
lab = fa['label'].values
Xtab = np.nan_to_num(fa[config.FEATURE_COLUMNS].astype(float).values, nan=-99.0)
gi = np.array([view[t] for t in fa['target']]); Vg, Vl = Xg[gi], Xl[gi]
classes = sorted(np.unique(lab)); yi = np.array([classes.index(v) for v in lab])
tr,te = train_test_split(np.arange(len(fa)), test_size=0.3, random_state=0, stratify=yi)
lgbm = lgb.LGBMClassifier(objective='multiclass', class_weight='balanced',
                          random_state=0, verbose=-1, **best); lgbm.fit(Xtab[tr], lab[tr])
tabp = lgbm.predict_proba(Xtab[te]); tabc = list(lgbm.classes_)
# CNN on the same train split
torch.manual_seed(0); m = cnn._build_model(len(classes))
opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
cnt = np.bincount(yi[tr], minlength=len(classes)).astype(float)
lf = nn.CrossEntropyLoss(weight=torch.tensor(cnt.sum()/np.maximum(cnt,1), dtype=torch.float32))
t3=lambda a: torch.tensor(a, dtype=torch.float32).unsqueeze(1)
Xg_tr,Xl_tr,y_tr=t3(Vg[tr]),t3(Vl[tr]),torch.tensor(yi[tr],dtype=torch.long)
rng=np.random.default_rng(0)
for _ in range(200):
    m.train(); perm=rng.permutation(len(tr))
    for s in range(0,len(tr),32):
        b=perm[s:s+32]
        if len(b) < 2: continue          # BatchNorm needs >=2 samples in train mode
        xg,xl=cnn._augment_batch(Xg_tr[b],Xl_tr[b],rng)
        opt.zero_grad(); lf(m(xg,xl), y_tr[b]).backward(); opt.step()
m.eval()
with torch.no_grad(): cnnp=torch.softmax(m(t3(Vg[te]),t3(Vl[te])),1).numpy()
tab_al=np.zeros((len(te),len(classes)))
for j,cl in enumerate(tabc): tab_al[:,classes.index(cl)]=tabp[:,j]
truth=lab[te]
for name,P in [('Tabular',tab_al),('CNN',cnnp)]:
    pred=np.array(classes)[P.argmax(1)]
    print(f'{name}: macro-F1={f1_score(truth,pred,average="macro"):.3f}')
best_w, best_f1 = 0.5, 0
for w in np.linspace(0,1,11):
    pred=np.array(classes)[(((1-w)*tab_al+w*cnnp)).argmax(1)]
    f=f1_score(truth,pred,average='macro')
    if f>best_f1: best_f1,best_w=f,w
print(f'best ensemble: w_cnn={best_w:.1f}  macro-F1={best_f1:.3f}')

## 7. Save + download the trained models
Pull `classifier.joblib`, `cnn_model.pt`, `train_features.parquet`, `train_views.npz` into the repo's `data/features/` so `predict.py` and the app use the big-data models.

In [ ]:
import shutil, os

ART = ['classifier.joblib', 'cnn_model.pt', 'train_features.parquet', 'train_views.npz']
# k-fold bagged models (predict() / predict_ensemble() average these)
for i in range(5):
    ART += [f'classifier_fold_{i}.joblib', f'cnn_fold_{i}.pt']

for f in ART:
    print(f, '-> exists:', (config.FEATURES_DIR / f).exists())

print(f'\nHeadline: CV macro-F1 (tuned) = {best_score:.3f} | ensemble macro-F1 = {best_f1:.3f}')

# --- Save to Google Drive (persists after session ends) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    dst = '/content/drive/MyDrive/exo_models'
    os.makedirs(dst, exist_ok=True)
    for f in ART:
        src = config.FEATURES_DIR / f
        if src.exists():
            shutil.copy(src, dst)
            print(f'saved {f} -> {dst}')
    print('All models saved to Google Drive.')
except Exception as e:
    print(f'Drive save failed ({e}), falling back to browser download...')
    from google.colab import files
    for f in ART:
        src = config.FEATURES_DIR / f
        if src.exists():
            print(f'downloading {f} ...')
            files.download(str(src))
    print('Check your Downloads folder. Copy all files to data/features/ in the repo.')